# Rossmann Sales — ML Experiment

Thí nghiệm huấn luyện LightGBM bằng **holdout theo thời gian** để đánh giá khả năng dự báo trên dữ liệu tương lai. Notebook sẽ:
- định vị project root an toàn dù chạy từ thư mục gốc hoặc `notebooks/`;
- kiểm tra phạm vi dữ liệu trước khi huấn luyện;
- so sánh mô hình với baseline không rò rỉ dữ liệu;
- lưu mô hình production và metrics khi chạy thành công.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / "backend").is_dir():
    root = root.parent

if not (root / "backend").is_dir():
    raise RuntimeError("Không tìm thấy project root (thư mục backend/) trong đường dẫn cha")

sys.path.insert(0, str(root / "backend"))

import pandas as pd
import matplotlib.pyplot as plt

from app.model.train_model import train_and_evaluate
from app.services.data_service import get_stores, get_train

VALIDATION_DAYS = 42
N_ESTIMATORS = 420

print(f"Project root: {root}")

## 1. Kiểm tra dữ liệu

Holdout sử dụng 42 ngày cuối. Mọi dữ liệu sau ngày cutoff được giữ riêng khỏi quá trình huấn luyện.

In [ ]:
train = get_train()
stores = get_stores()

overview = pd.DataFrame([{
    "store_count": stores["Store"].nunique(),
    "train_rows": len(train),
    "date_start": train["Date"].min().date(),
    "date_end": train["Date"].max().date(),
    "open_rows": int(train["Open"].eq(1).sum()),
    "positive_sales_rows": int((train["Open"].eq(1) & train["Sales"].gt(0)).sum()),
}])
display(overview)

if train["Date"].isna().any():
    raise ValueError("Dữ liệu train chứa Date bị thiếu")

## 2. Huấn luyện và đánh giá

Lần chạy này tạo validation LightGBM, tính baseline theo `(Store, DayOfWeek, Promo)`, rồi huấn luyện lại mô hình production trên toàn bộ lịch sử hợp lệ.

In [ ]:
metrics = train_and_evaluate(
    validation_days=VALIDATION_DAYS,
    n_estimators=N_ESTIMATORS,
)

comparison = pd.DataFrame.from_dict(
    {
        "baseline": metrics["baseline"],
        "lightgbm": metrics["validation"],
    },
    orient="index",
).rename_axis("model").reset_index()

comparison["mae_improvement_pct"] = (
    (comparison.loc[comparison["model"] == "baseline", "mae"].iloc[0] - comparison["mae"])
    / comparison.loc[comparison["model"] == "baseline", "mae"].iloc[0]
    * 100
)
comparison

## 3. Phân tích kết quả

- `MAE`, `RMSE`, `WAPE`: càng thấp càng tốt.
- `RMSPE`: chỉ tính trên các ngày có doanh số dương.
- Cột `mae_improvement_pct` dương cho biết mô hình giảm MAE so với baseline.

In [ ]:
display(comparison.style.format({
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "rmspe": "{:.5f}",
    "wape": "{:.5f}",
    "mae_improvement_pct": "{:+.2f}%",
}))

print("Validation period:", " to ".join(metrics["validation_dates"]))
print("Training period:", " to ".join(metrics["train_dates"]))

In [ ]:
importance = pd.DataFrame(metrics["important_features"])

fig, ax = plt.subplots(figsize=(9, 6))
importance.sort_values("importance").plot.barh(
    x="feature", y="importance", ax=ax, legend=False, color="#4C78A8"
)
ax.set_title("Top LightGBM Feature Importance")
ax.set_xlabel("Split importance")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## 4. Kết quả đã lưu

Khi chạy thành công, notebook đã cập nhật:
- `models/rossmann_sales_lgbm.joblib` — mô hình production cuối cùng;
- `results/evaluation_metrics.json` — metrics holdout và metadata.

Không chỉnh các cột `Sales` hoặc `Customers` trong đặc trưng đầu vào; business rule vẫn ép cửa hàng đóng cửa dự đoán bằng 0.